# Propensity score matching for outcomes in wake-up stroke treatment (presented at ISC21)

In [ ]:
import pandas as pd
from causalinference import CausalModel
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
data_path = './data/Fichier Wake-Up v03.xlsx'

## Outcome variables:
anybleeding_all infarct_growth	MRS01   TICI_2b-3 asymptomatique_ICH Symptomatic_ICH sytemic_bleeding

In [ ]:
outcome_var = 'sytemic_bleeding'

## Covariates

In [ ]:
# Max 14 covariates
covar_fields = ['Age_(calc.)', 'Sex', 'Discovery_to_groin', 'Onset_to_groin_puncture_(min.)',
                'NIH_on_admission',
                'MedHist_Hypertension', 'MedHist_Diabetes', 'MedHist_Hyperlipidemia',
                'MedHist_Atrial_Fibr.', 'wake_up', 'T6', 'CBF', 'Localisation', 'ASPECTS']

if outcome_var == 'infarct_growth':
    # 9 covariates for infarct growth
    covar_fields = [ 'Discovery_to_groin', 'Onset_to_groin_puncture_(min.)',  'NIH_on_admission', 'wake_up',
                     'T6', 'CBF', 'Localisation', 'ASPECTS', 'Age_(calc.)']

## Intervention : IVT with rTPA

In [ ]:
intervention_var = 'IVT_with_rtPA'

In [ ]:

data = pd.read_excel(data_path)

data.columns = data.columns.str.replace(' ','_')

# Encode mRS (0 or 1) vs (> 1)
data['MRS01'] = data['3M_mRS'].isin([0, 1])

# Encode wake-up
data['wake_up'] = data['Time_of_symptom_onset_known'] == "wake up"

# Encode infarct growth
data['infarct_growth'] = data['lesion_volume_ml'] - data['CBF']
data.loc[data['infarct_growth'] < 0, 'infarct_growth'] = 0

# Encode TICI
data['TICI_2b-3'] = data['TICI_2b-3'].astype('int')

# Retain only relevant variables
data = data[covar_fields + [outcome_var, intervention_var]]



In [ ]:
# Drop NaN (has to be done before one-hot-encoding)
data = data.dropna()

In [ ]:
# Clean localisation column
if 'Localisation' in data.columns:
    data['Localisation'] = data['Localisation'].str.replace('ICA','ACI')
    data['Localisation'] = data['Localisation'].str.replace('et','+')

# One-hot encoding for categorical vars
if 'Sex' in data.columns: data['Sex'] = data['Sex'] == 'Male'
if 'MedHist_Hypertension' in data.columns: data['MedHist_Hypertension'] = data['MedHist_Hypertension'] == 'yes'
if 'MedHist_Diabetes' in data.columns: data['MedHist_Diabetes'] = data['MedHist_Diabetes'] == 'yes'
if 'MedHist_Hyperlipidemia' in data.columns: data['MedHist_Hyperlipidemia'] = data['MedHist_Hyperlipidemia'] == 'yes'
if 'MedHist_Atrial_Fibr.' in data.columns: data['MedHist_Atrial_Fibr.'] = data['MedHist_Atrial_Fibr.'] == 'yes'
if 'Localisation' in data.columns:
    data['Localisation'] = pd.Categorical(data['Localisation'])
    data['Localisation'] = data['Localisation'].cat.codes

data['IVT_with_rtPA'] = data['IVT_with_rtPA'] == 'yes'

if outcome_var == 'any_bleeding' \
        or outcome_var == 'sytemic_bleeding' \
        or outcome_var == 'asymptomatique_ICH' \
        or outcome_var == 'Symptomatic_ICH' :
    data[outcome_var] = data[outcome_var] == 'yes'

data = data.astype(float)


In [ ]:
# Number treated
data[intervention_var].sum()


In [ ]:
# Number not treated
data[intervention_var].size - data[intervention_var].sum()

In [ ]:
# median and IQR of events in treatment group
print(data[data[intervention_var] == True][outcome_var].median())
data[data[intervention_var] == True][outcome_var].quantile(0.25), data[data[intervention_var] == True][outcome_var].quantile(0.75)

In [ ]:
# median and IQR of events in control group
print(data[data[intervention_var] == False][outcome_var].median())
data[data[intervention_var] == False][outcome_var].quantile(0.25), data[data[intervention_var] == False][outcome_var].quantile(0.75)


In [ ]:
# Number of events in treatment group
data[data[intervention_var] == True][outcome_var].sum()

In [ ]:
# Number of events in control group
data[data[intervention_var] == False][outcome_var].sum()

Defining model with regards to IVT as intervention

In [ ]:
intervention = data.IVT_with_rtPA.to_numpy()
outcome = data[outcome_var].to_numpy()

covars = data[covar_fields].to_numpy()

In [ ]:
causal = CausalModel(outcome, intervention, covars)
print(causal.summary_stats)

Estimate propensity scores

In [ ]:
causal.est_propensity()
print(causal.propensity)
print(causal.propensity.keys())

Match by propensity scores (nearest-neighbour)

In [ ]:
causal.est_via_matching(bias_adj=True)
print(f'Outcome variable: {outcome_var}')
print(causal.estimates)

In [ ]:
data['propensity_score'] = causal.propensity['fitted']

## Distribution of covariates with respect to treatment and outcome

In [ ]:
g = sns.pairplot(
    data,
    x_vars=covar_fields,
    y_vars=[outcome_var],
    hue=intervention_var,
    kind='scatter',
)
g.map(sns.kdeplot, alpha=.5, levels=4)

g.fig.suptitle(f'Distribution of covariates with respect to treatment and outcome ({outcome_var})', y=1.08)

plt.plot()

In [ ]:
# g.savefig(f'{outcome_var}_covariates_distribution.png')

In [ ]:
# sns.stripplot(x=outcome_var, y="propensity_score", hue='IVT_with_rtPA', data=data)
#

In [ ]:
# sns.stripplot(x="index", y="propensity_score", hue='IVT_with_rtPA', data=data.reset_index())

